# Multi-head self-attention from scratch

Build multi-head self-attention as an `nn.Module`, in the style of 2025 Round 2 Problem 2 and 2026 Round 2 Problem 1 (where the head size was not `d_model / n_heads`).

```python
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, d_head):
        # Your code
    def forward(self, x, key_padding_mask=None, causal=False):
        # Your code
```

**Parameters.** Create exactly these four layers, all with biases, and no other parameters:

| attribute | layer |
|---|---|
| `self.W_q`, `self.W_k`, `self.W_v` | `nn.Linear(d_model, n_heads * d_head)` |
| `self.W_o` | `nn.Linear(n_heads * d_head, d_model)` |

Do not assume `n_heads * d_head == d_model`.

**Forward.** `x` has shape `(B, T, d_model)`. Head `h` (for `h = 0, ..., n_heads - 1`) uses features `h * d_head` to `(h + 1) * d_head - 1` of `W_q(x)`, `W_k(x)` and `W_v(x)` as its queries, keys and values. Each head computes scaled dot-product attention with scale $1/\sqrt{d_{\text{head}}}$. The head outputs are concatenated in head order (head 0 first) into shape `(B, T, n_heads * d_head)` and passed through `W_o`. Return shape `(B, T, d_model)`.

- `key_padding_mask`: `None` or a `torch.bool` tensor of shape `(B, T)`, `True` at real tokens. Keys at `False` positions get weight 0 in every head and for every query.
- `causal`: if `True`, query `i` may attend only to keys `j <= i`.

Both masks can be given at once. No dropout. Python loops are allowed, but you do not need one over the heads: the reshape of u15-l04 (`view(B, T, n_heads, d_head)` then `transpose(1, 2)`) runs every head at once, as Round 2 graders expect.

**Example.**

```python
torch.manual_seed(0)
m = MultiHeadSelfAttention(d_model=8, n_heads=2, d_head=4)
y = m(torch.randn(3, 5, 8), causal=True)
y.shape                                      # torch.Size([3, 5, 8])
sum(p.numel() for p in m.parameters())       # 3 * (8 * 8 + 8) + (8 * 8 + 8) = 288
```

**Restrictions.** Only `import torch`, `import torch.nn as nn` and `import math` (anything inside `torch` is allowed). You may not use `nn.MultiheadAttention`, `F.scaled_dot_product_attention`, `multi_head_attention_forward`, `nn.TransformerEncoderLayer`, `nn.TransformerEncoder` or `nn.Transformer`.

**Constraints.** $B \le 16$, $T \le 256$, $d_{\text{model}} \le 256$, $n_{\text{heads}} \le 16$, $d_{\text{head}} \le 64$. Time limit 1 second per forward call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem2_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem2_submission.py
import math
import torch
import torch.nn as nn


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, d_head):
        super().__init__()
        # Your code

    def forward(self, x, key_padding_mask=None, causal=False):
        # Your code
        return x


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem2_submission as submission
importlib.reload(submission)
# e.g. submission.MultiHeadSelfAttention(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9XI1z2zaW/1dw7kxDpjIjKx/NKnWvTuo2ucnXJNnJ3tgaFhIhiTVFMgQV2+v1/e33fg8ACVKS46R7183IJAg8PLzv9wDs1V6a7I3F3vrg4X45HO0NxN5K1ZKarja/1GmdKbS9Wmd1ur9UMhFaZfN9Wdcqr9MiF/OqWAk9q2Q9W2JIprQucgcnGz7gGdJ8XStNraMhvSbpfJ7OCOQltTygBoJVXTbzPKdp3tMsR24SBlEk60zFuVwxQmVVTDO1GsV6PV2lWtteMsuKc5XE6aosqhoTnuzVRWVQW8l6uTehh6nMc5XwR54Q6+pMpmcyIyhJUcc0UbKe1bHsIINRMYa17fG8qM5lleD7h0rmmt5XqjrOZ0WiqpfyUlXbP/VaHYZxVhQlVjCXmVZghtJ1XKcrVazruKFlLRdmmQ0i0WopmXlYd2To1r5XSi9lyQ3eCKnP0nzBU+tiXc2YxkWVLtJcZuhbyqoeUWNdrdU1vc/TjPl5tbeUVU48j0pm4Cn/77nKSlVpUS9lLSyrxKKSCRoNb0TwB4uOHe5a7/4RRqf5aX5ku4tUi1mRA9F76zyt9b2f8OfnexYqvZdp8vM905uQiMSLWiRqnhJUAZKJ+TqfYZVaQHgSbozvGtxqeaboSZ3mrRwJQzLxBzX9EYnjfxy9evvy+L3IUl1jScqABTAt9LI4zwUNerfORfA51SlhFT4R7wGuFtWaplWfVXXJg2hpv8k0WxMTRCVTrQRexXlaL4UUBE/LhRLnVQrOCJIH8VTlY6HlpTgHuudS09ciXwxEDqgiWa9KsUyThLqnebmuNahnuJBbks6K8rJ5gfyAvvY1p/GXgqDmJVq/E8+KTE7Fs7d/10JWSmioUwVa5UDljhavRuJtVQyELkRWkJoAIomOFqs1EWWqQFQiUQENT+lzloIOU1r/klg5lzMSwwgzjaIhmCvFYg3+B2ZmTbAytV8vK7Y1pSKGUa+qWC+W2SVJSzYXhAthMSsqELpSIHot0jkQpFdWfLsOqy0gyvu3x8e/xr8dPfvw5p04xOxY8Gk+yyTNDjYExxczVUJSwvFpLui/kj6ZXiRQRHGVBCSLyUCs9ML1oXnzAkTOE9uC/1rmBujbAonzMrhwY8noeWMch6CmXuuc1p/mupb5TAUXA/M9+qByXVShN5ynVfW6ysVFlJBJny2DMJqVa/plNgeh6ax4meIFT3dcVUXlQTFL9mDlZSS1rCp5SXi362AjEiyKekBCmeOX5PPwdI8km2zj6Z7DTBOt63WZqQArp/5hxEMtLkxT6uM6AVY4EPPTvSsAvCaGazOXuNJkd9RFqWY1ycSV1/+apmsxm2WFvgmzgajqIjs8UPuPBkLax8cOX9Ksc0K5QVbq+pIWOs8KCcTQznN2P3iLWZj1YUnn5nHHcswCbPdw29rcJ15eMwExJNVk3tKaRhMeWRaEgiwFpND/dm6/+dPDksqULWGOMWk+b4BbQQa/s8wSkYhhyYUfSy78+IKnqoooxoQAcit5gT9yqoOF2BfnYRgCNi00/acSinyZGEL5tqhKl05s6MRnmZF9EEEmqwXMroLAiiv6M47uL67DDu/Vp7XMbiGVXaUyXYF0nrCkD0KmKCFBLrKy30/3mB0tkC3i4glI2+f/XCx80WD8Y0sHQm5DmXyqNhDAlXEfYxIGYMuLb2GQMb6iT/9RdVBDL2rqMCPVMasIswNPeic/eEKPI+2QDfzRSqSwgMMo5pAwjjvoLMj8gXN21juu0x0HtYtprHNZkiev/39Nc1bk6iuNMk3ppPICIsm+fS8kl5iw9nbxCZJ0RoRE4BL6+HQX18EOAINEqfLwA0V6noBZBD8Q+frobQIIOy4ETRFg8reLDuGJLYEciOm3013eiu6mi9ELnu9byS5Bdoaju6bATiTNJDqYhttsDQ1v7czW4Y2dmNpH5m1Xs7GCgbF2JNm5zysHhyF4wUsRr9akFBTdBPN8IO6SOSXNuHv37BxPDhMTOj4jDyDmOc88R4CKNVCMQ1HggtRL5hQyVov1ioJyciGizORMRS7qZK8PfSK7eNJqlgw5npUYgBknpidFZ3AeedBHyHzGkJQEmRasQwxVFMooSjZV8M+0DMwYnqMj4GxPGuGyRuSSEhuBjAsBfjpPEVkiZG7X8t1VKn4QB9dPxHlRnSGqlyy8AuyjeLSxl5bIhHxLYQTxaR5ohVCQsLotlZ/4JCbr6ofcWyJpznDJH5oUlQnEse6dZiQF7yaHwlRvCESFFIwSpJXGkjgOfyVnYkYTYAgt0kTdyFY4sEdwQNnF/nSdkCVFM8fgFJanSKDWJamTqM/TmQJAolBKHrqYWw0mnB1FNYsQWuYUL8F9AwKNWNO3SHzAh7SiZlpMdgdTwC+YyUBhlXTEqh4imiQIEWWX85jBqMpZmy+IUlJvH0wxSm3DEbveQ2GZKO4JP21oRTKGKFbQhmDkyx0xkab56dBC6pmiKeU0Z22THA3E2Yhm69pHRnnQa+ys5MukYDxBiBGTYeQ1MxkoYwsSuNgd5PDjTX9BrEd1UbDcUPKf1ONoNL+mKJ+kTD2xWVcjxCxMpHYMSKxz5PNXljR2HOeQlNg5Fm/5j6ekzJLS4DmJ1gIBrEkyryyfGBC040YFrfIFaadKHMPaBIc4mRQrovZckvjFTcdTLvastZRpEXOFoVPl+B0tpAGEWVs8SNloLNMsgdbNUBmhhDmFLqDnf71/85pwzVlzdJ0Qhqyr5WW9pJb9F8LNI36yJY44SaufxU/tHPSR3kkr/8Vt9c8bKT8pm5f+E2Mv6iydNk3mD7VE6zrN2uaiefxTF22RoJT10h+u00Uu22H6UvfKC+6lIr8wlbOzpoWkOEexwNQa9CylZeaqMXfUcJbW+5mibuIgehQd2JhOj1EnWqG2OGCTQtmFdT7O+JUyJSovTnM3RzRPM5Jo9xqc7hHaRaWQAdpKC0Wi0d2/52c5aji6yFBRKbgCQBOmJbgW3bWR4vPjd8ekOZYW0Vv6G8QxymBxHKKqhuEU1ZWyIsxOc6JKhM4RSauq6mBIykDhA8CEBHBb7YtrQeI78tWf5FgcPxiOuBL28uPRf7+Pnx69fn38K2FwdboX2/pmHGMxtDDHT/OqLtTMPlGcb55mxaokXE/3rk/zl2/evI1fv/n1+D2BC0hWot8K0lU8fCTBVebxSF/ms+bDS4ohnxEM8/ZeeS+/UpDZvv2ucvjnojq+KEO/xPLB1GK+VGWJJWV6qwAytl6RvSG3pRqN5VTRAfKrEYhsZvFsqWZnOtDVDCyuZdffvjP6jmiYnBXoWJwZ4qz0ggjzhMwiIkBT42O10fWdhjscDNlSIhxVXaWmrNhxUcYJnUy2RbN1pSiwYyqRlFB+TYh249D3l5SfX3AciqKc2owPTyzaY5IWLgtb7Mcwk5qH2xS5yI2duVIR/ubF9RjP1JuSn2uLoK2XwyUQvSIyzKQnvSI6aHQysYhOZdPOfrJ2LofLofFaMzCvfUr6EJtaqW2HiFPISLqbJqg/UyRWmUDBACYIpn+AjAZBdzrF7gEt54xjN5VFEGw7AZfJPZqvZL1aZ37LnKsjZKphm4n45zI7C8CMnuv2wnR0N/JsUoN+VtGEsugYMbrjTQdmSyoULweSOyEoX6G0YF4jmo0IicgxWFHkSeYIyRyArxhZw4hwvN03eryIZJLYOcLNzi1vvH6RLrMU7MacJ/sHky0DW+ZF6zJB1L1lrJ8kZjeT8Teyev3VEEOJV0xHW3in5bcKtYWWPMISkp7/AhX7FCRoPSp4pOtQYJsE9IbuIt43DeV1fi3Rj5zu9NfeEwjGA4p2K6ivPYt8E8A02b2q7Z22zdg6q/6crPiRLEuVJ4Gr+JhBp3vG5Bnz/p9dem2bhb3g0zR/Y73Y0XpxpOGDQlNb6Q2ICtvvlayxg7gh08YGfSVyhJgnkB5IBIkW1E7z76IAoyVG5NHKCoF+0Z8F0lM2sIE3Txhuty4/wKMQYRyoqzsDcccAcRoFBbxD81Heu7iDUuz19nriNvxRstiCvna4t8C+a3YQpzLXoO1saT0Eh4Ktg+Dys5KUxP6D/4ZPTDaanctLvW92fSmTobgwbQBBKC20Is8uB25OylxaVxMYKTBFJBtKIei4FJlaUGp+jrBJcM8oikLPmVFQJpICTDGAlynnl4YLgec0v2fn6Hlhg651viSG/xKB50u/F5240BMhmuGrRYcw0MLSB/ADHW6RHYIc/hUO54Wbo9mSbblMmHcXHxsliovSxJPdahu6GyXrqd5tFoyMkuTiF+FgI2X19EYEJnKyM5wMJ75wb1n+15CgnfWmxbNx21gyt37bgk2YxMLeWSg1Q40h1JVaqpyT2CZ25FH/3vXblV3fkKnPq5TAZJcxR7IBgaBEN0Y21Y3o3+Rqn/FMlJ5VKWcVoiRRJf1eCLsTzR1St32NhI3UVaGf5s36boVpirKKy1sjypvpuY7raaCiOG4+UMpng00Ap0hzzk59jnkIBHFqjvRTuZgPmV+zBBuVnqNkgoIVrCzXJq3Q0S/CMS9q7zSBFSww6M3baS36lpjYMeEdEuXtjyD6v77iWa/R3xF7RZF40ORiZGn7WS4yWVktPp80IaJbys6eo4mXEltKIQA/FE2X+y5cJ7mndhQcoqyQiQ4CRuIexAY6waWIPcAjn4VKRuCMHaVPgGiR8Ts4anA18VBs6oVJGNF0MsE7Uj3dvprCjBFZm0lDEtHwuiAW+PBPWmgToLMjFW0UHREsVFCfGKwm7C3wxloy8cNZl9K5eFZz3XFjWt9KcNUiYHLicIgOTHffyxspcSXPOSGdFtH7GuWTF28c8UyRJzJ/Avv2/sXvRy/fvRrYLN12JXMWc3HX7QJ7tqx7aAm0HA0dLt3smA/BtMUqYmeSVoqUzzAjIEz7YZbDUdUppqgcmi8+vHh1/C5+d3z0ctBg1wtzdKkgPd1SWITWGLUZU9lBdZK3TLwVeQfRjGi0hTlik2eqevOtp5vTWVg8IaYO8NMfSBpj+ukTfOaEYWIEf3NJrEOqilADis0wWJ7wGwg37NYmbNmlU5KwgmiVY2KsmVkjLDBbNba4dVGc8Z6CuHL8GEdDVICd6bKzPJVaNRUiUwNpS2JPXx4PhwctAl+1jF0Y240hRhM7MTghVlhG2SDoBn/UqPUG6G2EcnYF85I8Q6J4Jz6gBPzBcDgcT75ZkfkIZbeOw03++Q7Tx+0Vux1NWsvAfAq/zNyWVkmhTNBkDtuRdwGIaz9v/8olLL5OJ0/3zKE/aGHjMppzgM267TnCL+vewlO+xS5dMtBsL3vczxa2XJZnusAyuKODLoi3Zg++BkFDQInK3Bh2PMG2f5aVdnNEaa1WOuDTM3nXK7BVdfv+2ECRhEYwD/2CF1kJbEJySIIZe8Ezu2NIf7Umu2WEAtEKnyLI3dJ6BhfmOc3XyuffzDhYYw/HDOTk4XhC9pv3hQlbW6UWbB5P9yxo19mbDV9RDUbasy1NsfbWbPqgaRgNr79mT2zz4MM3eJ2/6nns5tymVf4mk2aZcNJSDpqKwHvz7MaGBW8Hm9jD6Dgb66zIF25fpm+zRZDm5pQZR/MDJBBzWQm3Mxj6RsBOzpsbvZr2X1xwgzPC6y1nVb7Kl/y7cPHPGfB+RXJLH9LCagQc8CqS4iTYtVE7EPe3uTcbzk5cQlY1hu2rXVEzwMSqh0gdgzZ2vSkqZaMyLYos2EDMfAOouiO5AFd3wDUjXIKy06Xwsea5cAmPsW5xjPwGG2Vjtz2AdMfs7DbOAnu6tnyGSwLtDqY5eOS/RTmLUp7v3sMb8H73wJxEHQh72s87gTOwh0WAcXO8nByCNezqQq5KGMhJm6WhapUYV02Qn9PvkiDTNIfD5uiUOe8vc5wK0nyw1mxi87JNwBhtv2AROJjh9vN4K0I/j16xC2Qbvh2MOwOOAyRN/8YRt15JL0vw+IR8w8f4E6x68Fzc5SUlIcHn9rMd7Z93tHM9lRfCn8LQj8AyidjIeWgshxFBMtc/ROktm0bxwl9SfCMrc4oI11+iKwy/7i3X9No8lWmObRKs6Fyli2Vtz29yXWBZ9oHaTptHQjcAdE48EqQtB0IxZpoSJFvfwno3VoGeOIePjg2Ila17duoKq1YeKeD6U5kaHsh50S3MHMvZ0uuC7daLAVjO+eGjBwNzbQjlmTZ7uqMF9uHNGru7qhfEvIuIDBbCHIvTR8Qd+VgEPldDR6LmlKUbNBDdfljtZi8bjQHVVjgbaWzEz8pbeN0hD7C5EL+Ijyf55GQ4iT6Q2efng8ltwIKhH73jKmoeIwIBdRuNP8PJTMPFmVxrmR1ylOSVb/4cMGU2+GM6PB2ID5S9MznNcWZTlSZN7u+YLtszTs/9EEhnMCVZSvqxtEpIDz+Ig9DoXdvzE+E7EJ8tNid24ZOTKIrIBmSTQdt+tqP9s9/u4UAwPxGlz6IaF6dKHFbfHw3EPiFxj014pD9VddBBp93rNpaSqKgDQxD6l6Bgdmi+wGd1N4wMsXuhiwP4vYNYVyk5tBb4Dshd0GCpr5+7J0HPk/HAavG4Rw/Nt7hUgiwpC/7HjhvYkszp3r457O9Nzlx3IYLBTxfzGgf4NaGdrg5Bz1/EZzuG8q6GePQc8PimY/fIL3WFIlhF8dShaTnwvBvlIFbOvyDgZzhtf3XdZLQ3EO/snOY6U5cUYyVJmi9ikMeEMBjVgNhgLQ80rf1gmrMFGwaYE2HBljO8/llbsyocRfSvqKA4GM+kVn2PbsSR0jmV7aRF3++vOILbER6ErQG19/8o8Mqt2Ccwiva4jhXQ5vgOGcV+KAElHzbVOzjULt8cmq6CyFeEyIO62exZfl7d9Rgl/3LdHtkzV06o+07D18DfDqe92POgvdjzsOfBGOZlywo/4OKUzBHVdMTpzB6zHg8EWRqa4v5APDTHrXg0xR4x7xhyD2tSyfHxtZcHLW32Gm56p7ZzDs9WQYmrWgr7jTDBHCWtcFiIHDUF/bopfZvrHgghRo8fMzksFmN2qvYgBWKIq/xatAC8kIEGdi5BMCWanibPMFdcsYKYNCyuU5V0iNSVvIORIco3rgl3SGjIfXIjwcGIfg8eQuIehvQb0DM1jNAw2qBBc0fFcYAx6bHg4e1Jw9dZrjepY27NxNBvsiU+JfpC4kjBsvIjpORG1JAdCGOfrHYf9Ofm3WqavLxkRfVruzu48Yil9IEFOGoUAQV/ROcbN58DO2SKqWI+mO3L6BeN3zyiTAuuO7ZBGI4ux0HrME5WEYUANkQbCLyddd4+27dJt0TYAuaobQdYfHNA2+fP/LwBENsRgBh1UMWAwjbtGuChYLqjwevcN7XE70eGG7e1tfd9ZI1ZdIY2ZPOIuz34F5Jb7cgW+Ne3PLcwkB1BM8bpdrbQCowxiX/jc6Y8ehONxz27Z6XygfNPIw5JO1eiLkYIdvYpeP6BrIK7Mng58vzOaJsttY6ER49pOKHePjoMx+Y2C3YpeJNY4kh5caZsDstXevm6i6CcLktxMJipyFtYjnaPNs1nG2r4FOQwxYmFNEH1o/DEBXLiJxdA8hWm4OQRW4+DSejCvcl2M2MIf5//PTLVU0agsSUcQhjfyfR+uAtjEtUtnN+G94834E2W7sEXkT4YOsPobKOPtS1pAxMCTjnb0lT4vZVsitEGI2CdUlxAucFANo6cQfzYJL6UUpgS0Q5tfshDb6vMj50yr6C1vrRGZXEejMIIPpLyUTnDJY8k2KiWWI/J/+cFse83+xeeygjL9sNhJqZp5gu5djLxsxiyv2zING4dobBlAdz/JK/kumx6Q76cfwN9Rw8fMaXujyyJ/7aDprAi3Jl/bkvYgyYS/YJjguGyl7MOoiGi69U0kUKOiSWyy5I2UW6DVx87rn3xTWQBee7EsHvX1/8LilA8BA==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
